# NOTE: Consider removing this step in the future and replace with code in model stream

In [1]:
import pandas as pd
import os
import numpy as np

# Get Home-Based Job percents from external process

In [2]:
wah_df = pd.read_csv(r"data\WAH-Chris-20251223\ControlTotal_WorkAtHome.csv")
wah_df

,;Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_GVED,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME
0,12023,1,2023,0.0711,0.0412,0.0679,0.0731,0.2093,0.1034,0.0759,...,0.0456,0.0334,0.0726,0.0283,0.0028,0.0274,0.0411,0.0934,0.1345,"Beaver County, Utah"
1,12024,1,2024,0.0666,0.0385,0.0635,0.0675,0.1959,0.0967,0.0711,...,0.0440,0.0323,0.0707,0.0274,0.0027,0.0265,0.0403,0.0886,0.1289,"Beaver County, Utah"
2,12025,1,2025,0.0617,0.0357,0.0589,0.0626,0.1816,0.0896,0.0659,...,0.0474,0.0349,0.0760,0.0295,0.0029,0.0285,0.0429,0.0811,0.1240,"Beaver County, Utah"
3,12026,1,2026,0.0592,0.0342,0.0565,0.0595,0.1741,0.0862,0.0632,...,0.0482,0.0354,0.0762,0.0299,0.0030,0.0289,0.0431,0.0771,0.1202,"Beaver County, Utah"
4,12027,1,2027,0.0571,0.0331,0.0545,0.0575,0.1681,0.0833,0.0610,...,0.0488,0.0357,0.0766,0.0303,0.0030,0.0292,0.0434,0.0741,0.1175,"Beaver County, Utah"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,10492061,1049,2061,0.0744,0.0431,0.0710,0.0865,0.2191,0.1036,0.0795,...,0.0384,0.0294,0.0661,0.0212,0.0025,0.0241,0.0458,0.1238,0.1696,Utah
1286,10492062,1049,2062,0.0747,0.0432,0.0713,0.0869,0.2199,0.1040,0.0798,...,0.0387,0.0297,0.0667,0.0214,0.0025,0.0243,0.0463,0.1243,0.1706,Utah
1287,10492063,1049,2063,0.0750,0.0434,0.0716,0.0873,0.2207,0.1045,0.0800,...,0.0391,0.0299,0.0673,0.0215,0.0025,0.0245,0.0467,0.1248,0.1715,Utah
1288,10492064,1049,2064,0.0753,0.0436,0.0718,0.0877,0.2215,0.1049,0.0803,...,0.0394,0.0302,0.0678,0.0217,0.0025,0.0247,0.0471,0.1253,0.1724,Utah


In [3]:
# FOR INCOMPLETE WAH DATA, SET WAH VALUES TO ZERO FOR ALL FORECAST YEARS

# 1. Identify unique counties
unique_fips = wah_df["CO_FIPS"].unique()

# 2. Determine existing year range and build new years
max_year = wah_df["YEAR"].max()
new_years = list(range(max_year + 1, 2066))   # 2065 inclusive

# If nothing to add, stop here
if len(new_years) > 0:

    # 3. Create a MultiIndex of all combinations
    new_index = pd.MultiIndex.from_product(
        [unique_fips, new_years],
        names=["CO_FIPS", "YEAR"]
    )

    # 4. Build an empty dataframe with those keys
    # Zero-fill all other numeric columns
    new_rows = pd.DataFrame(index=new_index).reset_index()

    # Identify numeric columns except CO_FIPS & YEAR
    num_cols = wah_df.select_dtypes(include="number").columns.difference(["CO_FIPS", "YEAR"])

    # Fill numeric with zeros
    for col in num_cols:
        new_rows[col] = 0

    # Add missing non-numeric columns too (if any)
    other_cols = wah_df.columns.difference(["CO_FIPS", "YEAR"] + list(num_cols))
    for col in other_cols:
        new_rows[col] = None

    # 5. Combine original + new rows
    wah_full_df = pd.concat([wah_df, new_rows], ignore_index=True)

    # 6. Sort neatly
    wah_full_df = wah_full_df.sort_values(["CO_FIPS", "YEAR"]).reset_index(drop=True)

else:
    wah_full_df = wah_df.copy()

wah_full_df['_Index'] = wah_full_df['CO_FIPS'] * 10000 + wah_full_df['YEAR']

wah_full_df


,;Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME,_Index
0,12023,1,2023,0.0711,0.0412,0.0679,0.0731,0.2093,0.1034,0.0759,...,0.0334,0.0726,0.0283,0.0028,0.0274,0.0411,0.0934,0.1345,"Beaver County, Utah",12023
1,12024,1,2024,0.0666,0.0385,0.0635,0.0675,0.1959,0.0967,0.0711,...,0.0323,0.0707,0.0274,0.0027,0.0265,0.0403,0.0886,0.1289,"Beaver County, Utah",12024
2,12025,1,2025,0.0617,0.0357,0.0589,0.0626,0.1816,0.0896,0.0659,...,0.0349,0.0760,0.0295,0.0029,0.0285,0.0429,0.0811,0.1240,"Beaver County, Utah",12025
3,12026,1,2026,0.0592,0.0342,0.0565,0.0595,0.1741,0.0862,0.0632,...,0.0354,0.0762,0.0299,0.0030,0.0289,0.0431,0.0771,0.1202,"Beaver County, Utah",12026
4,12027,1,2027,0.0571,0.0331,0.0545,0.0575,0.1681,0.0833,0.0610,...,0.0357,0.0766,0.0303,0.0030,0.0292,0.0434,0.0741,0.1175,"Beaver County, Utah",12027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,10492061,1049,2061,0.0744,0.0431,0.0710,0.0865,0.2191,0.1036,0.0795,...,0.0294,0.0661,0.0212,0.0025,0.0241,0.0458,0.1238,0.1696,Utah,10492061
1286,10492062,1049,2062,0.0747,0.0432,0.0713,0.0869,0.2199,0.1040,0.0798,...,0.0297,0.0667,0.0214,0.0025,0.0243,0.0463,0.1243,0.1706,Utah,10492062
1287,10492063,1049,2063,0.0750,0.0434,0.0716,0.0873,0.2207,0.1045,0.0800,...,0.0299,0.0673,0.0215,0.0025,0.0245,0.0467,0.1248,0.1715,Utah,10492063
1288,10492064,1049,2064,0.0753,0.0436,0.0718,0.0877,0.2215,0.1049,0.0803,...,0.0302,0.0678,0.0217,0.0025,0.0247,0.0471,0.1253,0.1724,Utah,10492064


In [4]:
# normalize WAH data to long format

# get list of HBJ columns
hbj_cols = [c for c in wah_full_df.columns if c.startswith("HBJ_")]

# filter to only necessary columns
wah_hbj_df = wah_full_df[['CO_FIPS', 'YEAR'] + hbj_cols]

# melt the HBJ columns to long format
wah_melted_df = (
    wah_hbj_df
      .melt(
          id_vars=[c for c in wah_hbj_df.columns if not c.startswith("HBJ_")],
          value_vars=hbj_cols,
          var_name="HBJFULL",
          value_name="HBJPCT"
      )
      .assign(
          TDM_SECTOR=lambda d: d["HBJFULL"].str.split("_").str[1]
      )
      .drop(columns=["HBJFULL"])
).sort_values(by=['CO_FIPS', 'YEAR', 'TDM_SECTOR']).reset_index(drop=True)

# select and reorder final columns
wah_melted_df = wah_melted_df[['CO_FIPS', 'YEAR', 'TDM_SECTOR', 'HBJPCT']].rename(columns={'TDM_SECTOR':'VARIABLE'})

wah_melted_df

,CO_FIPS,YEAR,VARIABLE,HBJPCT
0,1,2023,AGRI,0.0283
1,1,2023,CONS,0.0274
2,1,2023,FOOD,0.0181
3,1,2023,GVED,0.0456
4,1,2023,HLTH,0.0334
...,...,...,...,...
14185,1049,2065,MING,0.0026
14186,1049,2065,OFFI,0.0840
14187,1049,2065,OTHR,0.0684
14188,1049,2065,RETL,0.0285


# Get employment data

In [5]:
tdm_employment_by_county_df = pd.read_csv('intermediate/split_emp.csv')
tdm_employment_by_county_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,738,1
1,1,2023,CONS,278,1
2,1,2023,FOOD,517,1
3,1,2023,GVED,810,1
4,1,2023,HLTH,160,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


# Join tables and calculate home-based jobs

In [6]:
tdm_hbj_df = pd.merge(
    tdm_employment_by_county_df,
    wah_melted_df,
    on=['CO_FIPS', 'YEAR', 'VARIABLE'],
    how='inner'
)
tdm_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS,HBJPCT
0,1,2023,AGRI,738,1,0.0283
1,1,2023,CONS,278,1,0.0274
2,1,2023,FOOD,517,1,0.0181
3,1,2023,GVED,810,1,0.0456
4,1,2023,HLTH,160,1,0.0334
...,...,...,...,...,...,...
15604,9157,2065,MING,164,57,0.0023
15605,9157,2065,OFFI,14202,57,0.0749
15606,9157,2065,OTHR,54646,57,0.0586
15607,9157,2065,RETL,20356,57,0.0254


In [7]:
# do banker's rounding to calculate HBJ counts
tdm_hbj_df['HBJ'] = np.round(
    tdm_hbj_df['VALUE'] * tdm_hbj_df['HBJPCT']
).astype(int)

tdm_hbj_df['NONHBJ'] = (tdm_hbj_df['VALUE'] - tdm_hbj_df['HBJ']).astype(int)
tdm_hbj_df = tdm_hbj_df[['CO_FIPS_split','CO_FIPS','YEAR','VARIABLE','VALUE','HBJPCT','HBJ','NONHBJ']]
tdm_hbj_df

,CO_FIPS_split,CO_FIPS,YEAR,VARIABLE,VALUE,HBJPCT,HBJ,NONHBJ
0,1,1,2023,AGRI,738,0.0283,21,717
1,1,1,2023,CONS,278,0.0274,8,270
2,1,1,2023,FOOD,517,0.0181,9,508
3,1,1,2023,GVED,810,0.0456,37,773
4,1,1,2023,HLTH,160,0.0334,5,155
...,...,...,...,...,...,...,...,...
15604,9157,57,2065,MING,164,0.0023,0,164
15605,9157,57,2065,OFFI,14202,0.0749,1064,13138
15606,9157,57,2065,OTHR,54646,0.0586,3202,51444
15607,9157,57,2065,RETL,20356,0.0254,517,19839


In [8]:
tdm_hbj_df.to_csv('intermediate/split_emp_with_hbj_detail.csv', index=False)

## RUN CHECKS

In [9]:
# 1. Before HBJ: totals from tdm_employment_by_county_df
before_df = (
    tdm_employment_by_county_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(VALUE_BEFORE_HBJ=('VALUE', 'sum'))
)

# 2. After HBJ: totals from tdm_hbj_df
after_df = (
    tdm_hbj_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(
          VALUE_AFTER_HBJ=('HBJ', 'sum'),
          NONHBJ_AFTER_HBJ=('NONHBJ', 'sum')
      )
      .assign(
          TOTAL_AFTER_HBJ=lambda d: d['VALUE_AFTER_HBJ'] + d['NONHBJ_AFTER_HBJ']
      )
)

# 3. Combine and compute difference
tdm_hbj_check_df = (
    before_df
      .merge(after_df, on=['CO_FIPS_split', 'YEAR'], how='outer')
      .assign(
          DIFF=lambda d: d['VALUE_BEFORE_HBJ'] - d['TOTAL_AFTER_HBJ']
      )
)

# show where differences exist
tdm_hbj_check_df[tdm_hbj_check_df['DIFF'] != 0]


,CO_FIPS_split,YEAR,VALUE_BEFORE_HBJ,VALUE_AFTER_HBJ,NONHBJ_AFTER_HBJ,TOTAL_AFTER_HBJ,DIFF


### ONLY 2023 DATA FOR NOW

In [10]:
if tdm_hbj_check_df[(tdm_hbj_check_df['DIFF'] != 0)].size == 0:
    print("HBJ calculation check passed: No differences found for 2023 data.")
else:
    print("else")

HBJ calculation check passed: No differences found for 2023 data.


# Change to long format

In [11]:
long_nonhbj_df = tdm_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','NONHBJ','CO_FIPS']].rename(columns={'NONHBJ':'VALUE'})
long_nonhbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,717,1
1,1,2023,CONS,270,1
2,1,2023,FOOD,508,1
3,1,2023,GVED,773,1
4,1,2023,HLTH,155,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,13138,57
15606,9157,2065,OTHR,51444,57
15607,9157,2065,RETL,19839,57


In [12]:
long_hbj_df = tdm_hbj_df.groupby(['CO_FIPS_split','YEAR','CO_FIPS'], as_index=False).agg(VALUE=('HBJ','sum'))
long_hbj_df['VARIABLE'] = 'HBJ'
long_hbj_df = long_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]
long_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,HBJ,194,1
1,1,2024,HBJ,195,1
2,1,2025,HBJ,217,1
3,1,2026,HBJ,228,1
4,1,2027,HBJ,231,1
...,...,...,...,...,...
1414,9157,2061,HBJ,7904,57
1415,9157,2062,HBJ,8017,57
1416,9157,2063,HBJ,8131,57
1417,9157,2064,HBJ,8240,57


In [13]:
long_df = pd.concat([long_hbj_df,long_nonhbj_df]).sort_values(['CO_FIPS_split','YEAR','VARIABLE'])
long_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,717,1
1,1,2023,CONS,270,1
2,1,2023,FOOD,508,1
3,1,2023,GVED,773,1
0,1,2023,HBJ,194,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,13138,57
15606,9157,2065,OTHR,51444,57
15607,9157,2065,RETL,19839,57


In [14]:
long_df.to_csv('intermediate/split_emp_with_hbj.csv', index=False)